# Name That LLM — submit notebook

One notebook for Kaggle. Settings: **GPU T4 x2**, **Internet off**. Output: `submission.parquet`.

Current backend: word+char TF-IDF + LogisticRegression (class-weighted).
Qwen3.5-9B-Base and Kev-9B slots are wired for later; they fall back until adapters are attached.

- Metric: balanced accuracy (mean per-class recall)
- Validation: GroupKFold on `prompt_id` (skipped on the hidden rerun)
- Interactive `test.csv` has 3 rows; the scored rerun has ~4000
- Never generate tokens. Never download Hugging Face weights here.


In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline

LABELS = [
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "glm-5.3", "glm-5.3-flash",
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "kimi-k3", "qwen3.8-max",
]
LABEL_SET = set(LABELS)

IS_KAGGLE = Path("/kaggle/input").exists()
IS_RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN") is not None
DATA = next(
    p for p in (
        Path("/kaggle/input/competitions/name-that-llm"),
        Path("/kaggle/input/name-that-llm"),
        Path("../dataset/name-that-llm"),
        Path("dataset/name-that-llm"),
    )
    if (p / "train.csv").exists()
)
OUT = Path("/kaggle/working") if IS_KAGGLE else (DATA.resolve().parents[1] / "outputs")
OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA / "train.csv", dtype=str)
test = pd.read_csv(DATA / "test.csv", dtype=str)
train["text"] = train["text"].fillna("")
test["text"] = test["text"].fillna("")
assert set(train["model"]) == LABEL_SET, set(train["model"]) ^ LABEL_SET
assert test["id"].is_unique

LIGHT = (not IS_RERUN) and len(test) < 50
print(
    f"data={DATA} train={train.shape} test={test.shape} "
    f"models={train['model'].nunique()} prompts={train['prompt_id'].nunique()} "
    f"kaggle={IS_KAGGLE} rerun={IS_RERUN} light={LIGHT}"
)


In [ ]:
def build_pipeline():
    return Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)),
            ("char", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), max_features=35000, sublinear_tf=True)),
        ])),
        ("clf", LogisticRegression(C=2.0, max_iter=1000, class_weight="balanced")),
    ])


def grouped_cv(estimator, texts, y, groups, n_splits=5):
    oof = [None] * len(texts)
    fold_scores = []
    for fold, (tr, va) in enumerate(GroupKFold(n_splits).split(texts, y, groups), 1):
        model = clone(estimator)
        model.fit(texts.iloc[tr], y.iloc[tr])
        pred = model.predict(texts.iloc[va])
        for i, yhat in zip(va, pred):
            oof[i] = yhat
        ba = balanced_accuracy_score(y.iloc[va], pred)
        fold_scores.append(float(ba))
        print(f"fold {fold}: balanced_accuracy={ba:.4f}  n={len(va)}  prompts={groups.iloc[va].nunique()}")
    oof_ba = float(balanced_accuracy_score(y, oof))
    recalls = recall_score(y, oof, labels=LABELS, average=None, zero_division=0)
    print(f"\nOOF balanced_accuracy={oof_ba:.4f}")
    print("per-class recall (worst first):")
    for name, rec in sorted(zip(LABELS, recalls), key=lambda kv: kv[1]):
        print(f"  {name:24s} {rec:.3f}")
    return {"oof_balanced_accuracy": oof_ba, "fold_scores": fold_scores,
            "per_class_recall": {n: float(r) for n, r in zip(LABELS, recalls)}}


pipe = build_pipeline()
cv_report = None
if not IS_RERUN:
    cv_report = grouped_cv(pipe, train["text"], train["model"], train["prompt_id"])
    (OUT / "tfidf_cv.json").write_text(json.dumps(cv_report, indent=2))
else:
    print("competition rerun: skip GroupKFold, fit on all train")


In [ ]:
def find_dir(root: Path, hints, markers):
    if not root.exists():
        return None
    hints = tuple(h.lower() for h in hints)
    for marker in markers:
        for path in root.rglob(marker):
            if any(h in str(path).lower() for h in hints):
                return path.parent
    return None


def qwen_base_dir():
    root = Path("/kaggle/input") if IS_KAGGLE else Path("../hf_cache")
    return find_dir(root, ("qwen3.5-9b-base", "qwen-3.5-9b-base", "qwen35-9b-base"), ("config.json",))


def kev_adapter_dir():
    root = Path("/kaggle/input") if IS_KAGGLE else Path("../hf_cache")
    return find_dir(root, ("kev-9b", "kev9b"), ("adapter_config.json", "head.pt", "config.json"))


def infer_qwen_classifier(texts, model_dir):
    # Phase 2: language-only Qwen3.5-9B-Base, max_length=512, linear 18-way head, no generation.
    raise NotImplementedError(f"Qwen classifier not wired yet ({model_dir})")


def infer_kev_decision(texts, adapter_dir, base_dir):
    # Phase 3: Kev pointer head over LABELS. Use T4 DeltaNet fallback, not kev.serve fused kernels.
    raise NotImplementedError(f"Kev not wired yet ({adapter_dir}, base={base_dir})")


t0 = time.time()
qwen_dir = qwen_base_dir()
kev_dir = kev_adapter_dir()
status = {
    "kaggle": IS_KAGGLE,
    "rerun": IS_RERUN,
    "light": LIGHT,
    "n_test": int(len(test)),
    "qwen_base": str(qwen_dir) if qwen_dir else None,
    "kev_adapter": str(kev_dir) if kev_dir else None,
    "backend": None,
    "notes": [],
}

if LIGHT:
    status["notes"].append("tiny test.csv / not a rerun; 9B skipped on purpose")

if not LIGHT and qwen_dir is not None:
    try:
        infer_qwen_classifier(test["text"].tolist(), qwen_dir)
    except NotImplementedError as exc:
        status["notes"].append(str(exc))
    gc.collect()
if not LIGHT and kev_dir is not None:
    try:
        infer_kev_decision(test["text"].tolist(), kev_dir, qwen_dir)
    except NotImplementedError as exc:
        status["notes"].append(str(exc))
    gc.collect()

status["backend"] = "tfidf"
pipe.fit(train["text"], train["model"])
pred = pipe.predict(test["text"])
status["notes"].append("9B experts not implemented; TF-IDF predictions")


In [ ]:
sub = pd.DataFrame({"id": test["id"].astype(str), "model": list(pred)})
assert list(sub.columns) == ["id", "model"]
assert sub["id"].tolist() == test["id"].astype(str).tolist()
assert sub["id"].is_unique
assert sub["model"].isin(LABELS).all(), sorted(set(sub["model"]) - LABEL_SET)
assert sub["model"].notna().all()

out_path = OUT / "submission.parquet"
sub.to_parquet(out_path, index=False)
# Kaggle scores ./submission.parquet in the working directory
if IS_KAGGLE and out_path.resolve() != Path("submission.parquet").resolve():
    sub.to_parquet("submission.parquet", index=False)

status["seconds"] = round(time.time() - t0, 3)
status["output"] = str(out_path)
status["cv_balanced_accuracy"] = None if cv_report is None else cv_report["oof_balanced_accuracy"]
(OUT / "t4_stub_status.json").write_text(json.dumps(status, indent=2))
print(json.dumps(status, indent=2))
print(f"wrote {out_path} rows={len(sub)}")
sub.head()
